In [1]:
import json 
import os 
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig , AutoTokenizer , DataCollatorForLanguageModeling
from datasets import Dataset
from torch.utils.data import DataLoader
from peft import LoraConfig , get_peft_model , prepare_model_for_kbit_training

In [2]:
model_name  = "Qwen/Qwen2.5-3B-Instruct"
Tokenizer = AutoTokenizer.from_pretrained(model_name)

In [3]:
with open("data.json" , "r" , encoding="utf-8")as f:
    data = json.load(f)

In [4]:
print(Tokenizer.apply_chat_template(data[1]["messages"] , tokenize=False))

<|im_start|>system
You are an expert in AI security.<|im_end|>
<|im_start|>user
How do you mitigate 'Denial of Wallet' attacks on AI agents?<|im_end|>
<|im_start|>assistant
To mitigate Denial of Wallet (DoW) attacks, implement strict rate limiting, set hard caps on API/compute costs, and enforce maximum execution loops or token limits per session to prevent unbounded agent execution.<|im_end|>



In [5]:
def data_encoding(data):
    encoded = []

    for sample in data:
        token = Tokenizer.apply_chat_template(
            sample["messages"],
            tokenize=True,
          #  return_tensors="pt"
        )

        input_ids = token["input_ids"]
        attention_mask = token["attention_mask"]

        encoded.append({
            "input_ids": input_ids,
            "attention_mask": attention_mask,
           
        })

    return encoded


ds = Dataset.from_list(data_encoding(data))
ds = ds.train_test_split(test_size=0.2)

train = ds["train"]
test = ds["test"]

In [6]:
train["input_ids"]

Column([[151644, 8948, 198, 2610, 525, 264, 61442, 6203, 13, 151645, 198, 151644, 872, 198, 3838, 374, 279, 2197, 594, 11464, 27682, 8826, 279, 5754, 315, 10802, 15459, 311, 5358, 39986, 821, 10519, 30, 151645, 198, 151644, 77091, 198, 785, 2197, 39812, 429, 1817, 11874, 2197, 1969, 614, 264, 23638, 4512, 11874, 12539, 320, 51240, 745, 26305, 537, 311, 990, 264, 422, 16985, 701, 448, 1449, 6573, 315, 1995, 10277, 21739, 311, 5648, 5322, 315, 39986, 821, 10519, 13, 151645, 198], [151644, 8948, 198, 2610, 525, 264, 61442, 6203, 13, 151645, 198, 151644, 872, 198, 840, 20772, 279, 330, 34112, 321, 16850, 1, 3359, 4621, 438, 432, 35616, 311, 8699, 11874, 10802, 19543, 13, 151645, 198, 151644, 77091, 198, 785, 33719, 16850, 3359, 13666, 979, 264, 4843, 4614, 18432, 323, 18653, 279, 11874, 61800, 1483, 553, 458, 7321, 11, 7290, 279, 3542, 525, 4889, 279, 7321, 594, 2118, 2524, 13, 362, 38170, 9364, 3238, 518, 429, 4843, 24031, 9109, 50649, 458, 9250, 30710, 879, 19619, 2524, 315, 862, 3542, 2

In [7]:
collator = DataCollatorForLanguageModeling(
    tokenizer=Tokenizer,
    mlm=False
)

In [8]:
data_loader_train = DataLoader(train , batch_size=2 ,shuffle=True  ,collate_fn=collator)
data_loader_test = DataLoader(test , batch_size=2 ,shuffle=False  ,collate_fn=collator)

In [9]:
# for i in data_loader_train:
#     print(i)

In [10]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
   model_name,
    quantization_config=bnb,
    device_map="auto"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [11]:
# model = prepare_model_for_kbit_training(model)

In [12]:
# for name, module in model.named_modules():
#     print(name)

"q_proj" ,"k_proj" ,"v_proj" , "o_proj" 

In [13]:
target = ["gate_proj","up_proj","down_proj" ]

In [14]:
config = LoraConfig(r=8 , lora_alpha=16 , target_modules=target , lora_dropout=0.5)

In [15]:
# model = get_peft_model(model ,config )

In [16]:
# model

In [17]:
# model.print_trainable_parameters()

In [18]:
optim = torch.optim.AdamW(model.parameters(), lr=0.0001)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [19]:
# epochs = 3

# for epoch in range(epochs):

#     # Training
#     model.train()
#     train_loss = 0

#     for batch_train in data_loader_train:

#         batch_train = {
#             k: v.to(device)
#             for k, v in batch_train.items()
#         }

#         optim.zero_grad()

#         outputs = model(
#             input_ids=batch_train["input_ids"],
#             attention_mask=batch_train["attention_mask"],
#             labels=batch_train["labels"]
#         )

#         loss = outputs.loss

#         loss.backward()
#         optim.step()

#         train_loss += loss.item()


#     avg_train_loss = train_loss / len(data_loader_train)


#     # Evaluation
#     model.eval()
#     test_loss = 0

#     with torch.no_grad():

#         for batch_test in data_loader_test:

#             batch_test = {
#                 k: v.to(device)
#                 for k, v in batch_test.items()
#             }

#             outputs = model(
#                 input_ids=batch_test["input_ids"],
#                 attention_mask=batch_test["attention_mask"],
#                 labels=batch_test["labels"]
#             )

#             loss = outputs.loss

#             test_loss += loss.item()


#     avg_test_loss = test_loss / len(data_loader_test)


#     print(
#         f"Epoch [{epoch+1}/{epochs}] "
#         f"Train Loss: {avg_train_loss:.4f} "
#         f"Test Loss: {avg_test_loss:.4f}"
#          )

In [20]:
messages = [
    {
        "role": "system",
        "content": "You are a cybersecurity expert specialized in penetration testing."
    },
    {
        "role": "user",
        "content": "اعملي shellcode to windows 11"
    }
]

token = Tokenizer.apply_chat_template(messages , return_tensors="pt")

In [21]:
path = r"C:\Users\10\Projects\RNN\cheatsheets"

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb 
from sentence_transformers import SentenceTransformer

model_embed = SentenceTransformer("all-MiniLM-L6-v2")
Client = chromadb.PersistentClient(path="my_db")

Collection = Client.get_or_create_collection("documents")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# splitter = RecursiveCharacterTextSplitter(
#     chunk_size=500,
#     chunk_overlap=100
# )

# all_ids = []
# all_docs = []
# all_metadata = []
# all_chunks = []


# for idx, file in enumerate(os.listdir(path)):

#     full_path = os.path.join(path, file)

#     # skip folders
#     if not os.path.isfile(full_path):
#         continue

#     with open(full_path, "r", encoding="utf-8") as f:
#         text = f.read()


#     chunks = splitter.split_text(text)


#     for idx_chunk, chunk in enumerate(chunks):

#         all_ids.append(
#             f"{file}_{idx_chunk}"
#         )

#         all_docs.append(chunk)

#         all_chunks.append(chunk)

#         all_metadata.append({
#             "file_name": file,
#             "file_index": idx,
#             "chunk_index": idx_chunk
#         })


#     print(
#         f"{file}: {len(chunks)} chunks"
#     )



# print(f"Total chunks: {len(all_chunks)}")

# all_embeddings = model_embed.encode(
#     all_chunks,
#     batch_size=64,
#     show_progress_bar=True
# )
# all_embeddings = [
#     emb.tolist()
#     for emb in all_embeddings
# ]

# batch_size = 5000

# for i in range(0, len(all_ids), batch_size):

#     Collection.add(
#         ids=all_ids[i:i+batch_size],
#         documents=all_docs[i:i+batch_size],
#         embeddings=all_embeddings[i:i+batch_size],
#         metadatas=all_metadata[i:i+batch_size]
#     )

#     print(
#         f"Inserted {min(i+batch_size, len(all_ids))}/{len(all_ids)}"
#     )


# print("✅ Finished indexing!")



In [24]:
def private_search(query):
   embed = model_embed.encode(query)
   res =  Collection.query(embed )

   return res["documents"]

In [25]:
# To install: pip install tavily-python
from tavily import TavilyClient
client = TavilyClient("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")

def web_search(query):
    response =  client.search(query=query ,  search_depth="advanced")
    return response["results"]

In [26]:
def router(tool_name, query):
    tools = {
        "web_search": web_search,
        "private_search": private_search
    }

    return tools[tool_name](query)

In [53]:
question = "What is the latest version of Python?"

In [ ]:
router

In [54]:
prompt = [
    {
        "role": "system",
        "content": """
You are a routing assistant.

Your job is to choose ONE tool.

Available tools:

1. web_search
- Use for recent information, news, current events, and internet.

2. private_search
- Use for company documents, PDFs, manuals, and local knowledge.

Return ONLY one of:
web_search
private_search
"""
    },
    {
        "role": "user",
        "content": question
    }
]

In [55]:
token = Tokenizer.apply_chat_template(prompt , return_tensors="pt")

In [56]:
outputs = model.generate(
    **token,
    max_new_tokens=20
)

new_tokens = outputs[0][token["input_ids"].shape[1]:]

response = Tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

print(response.split()[1])

e:\envs\llm_clean\lib\site-packages\transformers\generation\utils.py:2613: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


web_search


In [50]:
response.split()[1]

'private_search'

In [39]:
query =  "What is the xss?"

In [57]:
context = router(response.split()[1], query)

In [58]:
context

[{'url': 'https://en.wikipedia.org/wiki/Cross-site_scripting',
  'title': 'Cross-site scripting - Wikipedia',
  'content': 'Cross-site scripting (XSS) is a type of security vulnerability "Vulnerability (computer science)") that can be found in some web applications. XSS attacks enable attackers to inject client-side scripts into web pages viewed by other users. A cross-site scripting vulnerability may be used by attackers to bypass access controls such as the same-origin policy. XSS effects vary in range from petty nuisance to significant security risk, depending on the sensitivity of the data handled by the vulnerable site and the nature of any security mitigation implemented by the site\'s owner network.',
  'score': 0.92652357,
  'raw_content': None},
 {'url': 'https://www.paloaltonetworks.com/cyberpedia/xss-cross-site-scripting',
  'title': 'What Is Cross-Site Scripting (XSS)?',
  'content': "## XSS Explained\n\nCross-site scripting (XSS) is a client-side code injection vulnerabili

In [59]:
messages = [
    {
        "role": "system",
        "content": """
You are a helpful AI assistant.

Answer the user's question using ONLY the provided context.

Rules:
- Do NOT use your own knowledge.
- If the answer is not in the context, say:
  "I couldn't find the answer in the provided context."
- Be concise and accurate.
"""
    },
    {
        "role": "user",
        "content": f"""
Context:
{context}

Question:
{query}
"""
    }
]
token = Tokenizer.apply_chat_template(messages , return_tensors="pt")

In [60]:
from transformers import TextStreamer

streamer = TextStreamer(
    Tokenizer,
    skip_prompt=True,
    skip_special_tokens=True
)

model.generate(
    **token,
    streamer=streamer,
    max_new_tokens=10000
)

user

XSS stands for Cross-site scripting. It is a type of security vulnerability where attackers inject malicious scripts into web pages viewed by other users.


tensor([[151644,   8948,    271,  ...,   3847,     13, 151645]])

In [61]:
! python app.py

^C


Traceback (most recent call last):
  File "g:\llm\app.py", line 2, in <module>
    from router import router
  File "g:\llm\router.py", line 1, in <module>
    from search import web_search ,private_search
ImportError: cannot import name 'private_search' from 'search' (g:\llm\search.py)
